# SDE-Seg research workflow
Select a GPU runtime, then run these cells in order. This notebook uses your existing `MyDrive/LA.zip` and preserves `MyDrive/sde-seg-run/checkpoints`. Corrected experiments are stored under `MyDrive/sde-seg-run/research-v2` because the previous trainer did not select checkpoints on dev EER. A GPU session must be started by you; this file does not claim that training has run.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
WORK = Path('/content/sde-seg')
RUN = DRIVE / 'sde-seg-run/research-v2'
RUN.mkdir(parents=True, exist_ok=True)
# Set this if LA.zip is in a different Drive folder.
LA_ZIP = DRIVE / 'LA.zip'
print('Existing checkpoints:', list((DRIVE/'sde-seg-run/checkpoints').glob('*.pt')))


In [ ]:
# Upload the source bundle produced by python scripts/package_colab.py.
from google.colab import files
uploaded = files.upload()
name = 'sde-seg-colab.zip'
if name not in uploaded:
    raise FileNotFoundError('Upload sde-seg-colab.zip')
with zipfile.ZipFile(name) as archive:
    destination = Path('/content').resolve()
    for member in archive.namelist():
        if not (destination/member).resolve().is_relative_to(destination):
            raise ValueError('Unsafe archive member')
    archive.extractall(destination)
os.chdir(WORK)
os.environ['PYTHONPATH'] = str(WORK)
def run(*args):
    subprocess.run([str(x) for x in args], check=True)
run(sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun')
print(torch.cuda.get_device_name())
run(sys.executable, '-m', 'pytest', '-q')


In [ ]:
# Extract the provided LA archive to local disk, keeping the Drive source intact.
if not LA_ZIP.is_file():
    raise FileNotFoundError(f'Set LA_ZIP to the actual archive path: {LA_ZIP}')
LA_ROOT = WORK/'data/raw/LA'
LA_ROOT.mkdir(parents=True, exist_ok=True)
if not list(LA_ROOT.rglob('ASVspoof2019.LA.cm.train.*.txt')):
    with zipfile.ZipFile(LA_ZIP) as archive:
        for member in archive.namelist():
            if not (LA_ROOT/member).resolve().is_relative_to(LA_ROOT.resolve()):
                raise ValueError('Unsafe LA archive member')
        archive.extractall(LA_ROOT)
# Inspect train/dev only; eval is held until final evaluation.
run(sys.executable, 'scripts/check_data.py', '--data-root', LA_ROOT,
    '--output', RUN/'logs/data_check.json')


In [ ]:
# End-to-end smoke on real LA data, isolated from research checkpoints.
run(sys.executable, '-m', 'sdeseg.experiment', '--data-root', LA_ROOT,
    '--results', RUN, '--smoke-only')


## Cross-domain data
No cross-domain data is downloaded automatically. Place the official In-The-Wild and ML-ITW audio and metadata in local Colab storage. Inspect the dataset card and file tree first. Use `scripts/prepare_manifest.py --inspect --metadata ... --audio-root ...`, then provide the actual column names and exact label map with a mapping rationale. The resulting `.inspection.json` records labels, languages (when supplied), counts, duration and the decision. Copy these summaries into REPORT.md before evaluation. See README.md for commands.

Set both manifest paths below for the full paper experiment. `LA_ONLY=True` explicitly runs an incomplete LA-only experiment and leaves cross-domain results N/A.


In [ ]:
ITW_MANIFEST = None  # e.g. WORK/'data/manifests/itw.csv'
MLITW_MANIFEST = None  # e.g. WORK/'data/manifests/mlitw.csv'
LA_ONLY = False
args = [sys.executable, '-m', 'sdeseg.experiment', '--data-root', LA_ROOT, '--results', RUN]
if LA_ONLY:
    args += ['--la-only']
else:
    if not ITW_MANIFEST or not MLITW_MANIFEST:
        raise ValueError('Prepare both inspected manifests or explicitly set LA_ONLY=True')
    args += ['--itw-manifest', str(ITW_MANIFEST), '--mlitw-manifest', str(MLITW_MANIFEST)]
# Resumes matching runs after a Colab disconnect. All models are selected on dev.
# Final evaluation is journaled and will not silently overwrite prior tests.
run(*args)


In [ ]:
run(sys.executable, 'scripts/make_tables.py', '--results', RUN)
import pandas as pd
for name in ('table3_parameters.csv', 'table4_eer.csv', 'latency.csv'):
    display(pd.read_csv(RUN/'tables'/name))


In [ ]:
# Download a copy; persistent checkpoints remain on Drive.
import shutil
local = Path('/content/sde-seg-results')
local.mkdir(exist_ok=True)
for folder in ('logs', 'tables', 'figures'):
    if (RUN/folder).exists():
        shutil.copytree(RUN/folder, local/folder, dirs_exist_ok=True)
archive = shutil.make_archive('/content/sde-seg-results', 'zip', local)
files.download(archive)
